# Try regression instead of classification

The binary classification approach was kind of pointless - 88% baseline means we're barely learning anything.

Let's just predict the actual speedup number instead.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
df = pd.read_csv('../data/processed/dataset.csv')
print(f"{len(df)} loops")
df['speedup'].describe()

In [ ]:
# quick look at distribution
plt.hist(df['speedup'], bins=30)
plt.axvline(1.0, color='red', linestyle='--')
plt.xlabel('Speedup')
plt.show()

## Features

In [ ]:
# add some ratio features (from the other notebook)
df['memory_ops'] = df['num_load_instructions'] + df['num_store_instructions']
df['memory_ratio'] = df['memory_ops'] / (df['num_instructions'] + 1)
df['compute_ratio'] = df['num_arithmetic_ops'] / (df['num_instructions'] + 1)
df['trip_count_log'] = np.log10(df['estimated_trip_count'].replace(-1, 1000000) + 1)

# TODO: try more feature engineering

In [ ]:
feature_cols = [
    'num_instructions', 'num_load_instructions', 'num_store_instructions',
    'num_branches', 'num_arithmetic_ops', 'nesting_depth',
    'memory_ratio', 'compute_ratio', 'trip_count_log'
]

X = df[feature_cols].fillna(0)
y = df['speedup']

print(f"Features: {len(feature_cols)}")
print(f"Speedup range: {y.min():.3f} - {y.max():.3f}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## Models

In [ ]:
# try a few models
models = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"{name}:")
    print(f"  MAE: {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²: {r2:.4f}")
    print()
    
    results[name] = {'model': model, 'pred': y_pred, 'mae': mae, 'r2': r2}

## Visualize predictions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    ax = axes[idx]
    ax.scatter(y_test, res['pred'], alpha=0.6)
    
    # perfect prediction line
    lims = [min(y_test.min(), res['pred'].min()), max(y_test.max(), res['pred'].max())]
    ax.plot(lims, lims, 'r--', alpha=0.5)
    
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f"{name} (MAE: {res['mae']:.3f})")

plt.tight_layout()
plt.show()

## Feature importance

In [ ]:
# random forest shows which features matter
rf = results['Random Forest']['model']
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

plt.barh(importances['feature'], importances['importance'])
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.show()

## Error analysis

In [ ]:
# look at where we're most wrong
best = results['Random Forest']
errors = np.abs(y_test.values - best['pred'])

plt.hist(errors, bins=20)
plt.xlabel('Absolute Error')
plt.ylabel('Count')
plt.title('Prediction Error Distribution')
plt.show()

print(f"Mean error: {errors.mean():.4f}")
print(f"Max error: {errors.max():.4f}")

## Save model

In [ ]:
import pickle

best_model = results['Random Forest']['model']

with open('../models/regression_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("Model saved")

## Notes

- MAE around 0.05-0.08 seems reasonable
- R² could be better - need more data or better features
- Random Forest > Linear (non-linear relationships)
- TODO: try predicting unroll factor (2x, 4x, 8x) instead of speedup